# 151 공통이력 두 상대변화 전문가의 고정 OR와 안쪽 선택 검증
149/150후보를재실행. 상대온도6 mean과동시변화강도4LR. 각판정선innernormal99%. 고정OR는외부결과로고른개발후보, inner_selected는해당외부Run제외선택. 새비라벨점수는만들지않음.

In [1]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.linear_model import Ridge,LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix,precision_recall_curve,average_precision_score
from scipy.special import expit
root=Path.cwd();t=root/'reports/tables'
d=pd.read_csv(t/'rg3_shortshot_148_common_labeled_features.csv',parse_dates=['TimeStamp'])
g=d.evaluation_run.to_numpy();y=d.type.eq('미성형').to_numpy(int);use=d.type.isin(['정상','미성형']).to_numpy()
sensors=[c for c in d if c.endswith('__delta1')];sensors=[c.replace('__delta1','') for c in sensors]
assert len(d)==591 and y.sum()==8 and use.sum()==574
def grids(ref):
 steps={}
 for v in sensors:
  values=np.unique(np.round(d.loc[ref[y[ref]==0],v].dropna().to_numpy(),2));diff=np.diff(values)
  steps[v]=max(float(np.median(diff[diff>.0001])) if (diff>.0001).any() else .01,.01)
 return steps
def base_features(ref,q):
 steps=grids(ref);dz={};vz={};sr={}
 for v in sensors:
  scale=np.maximum(d.loc[q,v+'__past10_std'].to_numpy(float),steps[v])
  dz[v]=d.loc[q,v+'__delta1'].to_numpy(float)/scale
  vz[v]=d.loc[q,v+'__dev10'].to_numpy(float)/scale
  sr[v]=d.loc[q,v+'__current_std10'].to_numpy(float)/np.maximum(d.loc[q,v+'__past30_std'].to_numpy(float),steps[v])
 return {
  'shared_std24':d.loc[q,[v+'__current_std10' for v in sensors]].to_numpy(float),
  'relative_thermal6':np.column_stack([vz['Hopper_Temperature'],vz['Barrel_Temperature_4'],dz['Barrel_Temperature_5']-dz['Hopper_Temperature'],dz['Barrel_Temperature_3']-dz['Mold_Temperature_4'],vz['Mold_Temperature_3'],sr['Barrel_Temperature_5']]),
  'relative_supply6':np.column_stack([dz['Plasticizing_Time'],dz['Cushion_Position']-dz['Plasticizing_Position'],sr['Cushion_Position'],sr['Average_Back_Pressure'],sr['Plasticizing_Time'],sr['Hopper_Temperature']]),
  'relative_pressure6':np.column_stack([dz['Max_Injection_Pressure'],dz['Max_Switch_Over_Pressure'],dz['Max_Injection_Speed'],dz['Filling_Time'],vz['Max_Injection_Pressure'],vz['Cushion_Position']]),
  'relative_std24':np.column_stack([np.log1p(sr[v]) for v in sensors]),
  'relative_joints6':np.maximum(0,np.column_stack([
    np.minimum(-vz['Hopper_Temperature'],dz['Barrel_Temperature_5']),
    np.minimum(dz['Barrel_Temperature_3'],-dz['Mold_Temperature_4']),
    np.minimum(np.maximum(1-sr['Hopper_Temperature'],0),np.maximum(sr['Max_Injection_Speed']-1,0)),
    np.minimum(-vz['Hopper_Temperature'],-vz['Barrel_Temperature_4']),
    np.minimum(np.maximum(sr['Cushion_Position']-1,0),np.maximum(1-sr['Plasticizing_Time'],0)),
    np.minimum(np.maximum(sr['Barrel_Temperature_5']-1,0),np.maximum(sr['Average_Back_Pressure']-1,0))])),
  '_dz':dz}
def make_features(ref,q):
 tr=base_features(ref,ref);te=base_features(ref,q)
 normal=y[ref]==0;rr=[];rq=[]
 # Same idea as gas normal-relation residuals, but on normalized changes rather than absolute levels.
 relations=[('Max_Injection_Pressure',['Max_Injection_Speed','Filling_Time']),('Plasticizing_Time',['Max_Screw_RPM','Average_Back_Pressure']),('Cushion_Position',['Plasticizing_Position','Max_Injection_Pressure']),('Mold_Temperature_4',['Mold_Temperature_3','Barrel_Temperature_5'])]
 for target,parents in relations:
  a=np.column_stack([tr['_dz'][v] for v in parents]+[tr['_dz'][target]])
  b=np.column_stack([te['_dz'][v] for v in parents]+[te['_dz'][target]])
  imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(a[normal])
  a=imp.transform(a);b=imp.transform(b)
  m=Ridge(alpha=10).fit(a[normal,:-1],a[normal,-1])
  scale=max(float(np.std(a[normal,-1]-m.predict(a[normal,:-1]))),.1)
  rr.append((a[:,-1]-m.predict(a[:,:-1]))/scale);rq.append((b[:,-1]-m.predict(b[:,:-1]))/scale)
 tr.pop('_dz');te.pop('_dz');tr['change_residual4']=np.column_stack(rr);te['change_residual4']=np.column_stack(rq)
 return tr,te
old_make_features=make_features
def make_features(ref,q):
 tr,te=old_make_features(ref,q)
 for values in [tr,te]:
  joint=values['relative_joints6']
  values['relative_intensity4']=np.column_stack([np.nanmax(joint,axis=1),np.sort(joint,axis=1)[:,-2],np.nanmean(joint,axis=1),(joint>1).sum(axis=1)])
  values['abs_thermal6']=np.abs(values['relative_thermal6'])
 keep=['relative_thermal6','abs_thermal6','relative_std24','relative_joints6','relative_intensity4']
 return {k:tr[k] for k in keep},{k:te[k] for k in keep}
memo={}
def predict(ref,q):
 key=(tuple(ref),tuple(q))
 if key in memo:return memo[key]
 assert set(g[ref]).isdisjoint(g[q])
 tr,te=make_features(ref,q);yy=y[ref];normal=yy==0;out={}
 for view in ['relative_thermal6','relative_intensity4']:
  a=np.where(np.isfinite(tr[view]),tr[view],np.nan);b=np.where(np.isfinite(te[view]),te[view],np.nan)
  imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(a[normal]);a=imp.transform(a);b=imp.transform(b)
  sc=RobustScaler().fit(a[normal]);a=np.clip(sc.transform(a),-12,12);b=np.clip(sc.transform(b),-12,12)
  lr=LogisticRegression(C=.1,class_weight='balanced',max_iter=3000).fit(a,yy)
  if view=='relative_intensity4':out['joint']=lr.predict_proba(b)[:,1];continue
  scores=[lr.predict_proba(b)[:,1]]
  m=SVC(C=1.,class_weight='balanced',gamma='scale').fit(a,yy);scores.append(expit(m.decision_function(b)))
  for cls in [RandomForestClassifier,ExtraTreesClassifier]:
   pp=[]
   for seed in [42,7]:
    m=cls(n_estimators=80,max_depth=4,min_samples_leaf=3,class_weight='balanced',random_state=seed,n_jobs=2).fit(a,yy)
    pp.append(m.predict_proba(b)[:,1])
   scores.append(np.mean(pp,axis=0))
  out['thermal']=np.mean(scores,axis=0)
 memo[key]=out
 return out
def threshold(yy,s,limit=.03):
 p,r,c=precision_recall_curve(yy,s);f=2*p[:-1]*r[:-1]/np.maximum(p[:-1]+r[:-1],1e-12);opts=[]
 for j,cut in enumerate(c):
  fp=int(((s>=cut)&(yy==0)).sum())
  if fp<=int((yy==0).sum()*limit):opts.append((f[j],r[j],-fp,cut))
 return max(opts)[-1] if opts else float(np.max(s)+1e-6)
records=[];choices=[]
def stats(yy,pr):
 tn,fp,fn,tp=confusion_matrix(yy,pr,labels=[0,1]).ravel()
 return int(tp),int(fp),int(fn),int(tn),2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0
for block in pd.unique(g):
 ref=np.flatnonzero((g!=block)&use);q=np.flatnonzero(g==block)
 outer=predict(ref,q);inner={n:np.full(len(ref),np.nan) for n in outer}
 for ib in pd.unique(g[ref]):
  mask=g[ref]==ib;pred=predict(ref[~mask],ref[mask])
  for n in inner:inner[n][mask]=pred[n]
 cuts={n:float(np.quantile(inner[n][y[ref]==0],.99,method='higher')) for n in inner}
 ip={n:inner[n]>=cuts[n] for n in inner};op={n:outer[n]>=cuts[n] for n in outer}
 ip['OR']=ip['thermal']|ip['joint'];op['OR']=op['thermal']|op['joint']
 st={n:stats(y[ref],p) for n,p in ip.items()}
 chosen=max(st,key=lambda n:(st[n][4],st[n][0],-st[n][1],n=='OR'))
 choices.append(dict(block=block,selected=chosen,inner_thermal_F1=st['thermal'][4],inner_joint_F1=st['joint'][4],inner_OR_F1=st['OR'][4],thermal_threshold=cuts['thermal'],joint_threshold=cuts['joint']))
 for n,pr in [('fixed_thermal',op['thermal']),('fixed_joint',op['joint']),('fixed_OR',op['OR']),('inner_selected',op[chosen])]:
  for j,i in enumerate(q):records.append(dict(model=n,TimeStamp=d.TimeStamp.iloc[i],run_id=block,type=d.type.iloc[i],actual=int(y[i]),prediction=int(pr[j]),thermal=int(op['thermal'][j]),joint=int(op['joint'][j])))
 print('completed',block,chosen,flush=True)
p=pd.DataFrame(records);rows=[]
for n,z in p.groupby('model'):
 z=z[z.type.isin(['정상','미성형'])];tp,fp,fn,tn,f=stats(z.actual,z.prediction)
 rows.append(dict(model=n,TP=tp,FP=fp,FN=fn,TN=tn,F1=f))
m=pd.DataFrame(rows)
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'choices')]:obj.to_csv(t/f'rg3_shortshot_151_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False));print(pd.DataFrame(choices).to_string(index=False))


C:\Users\hayeon\AppData\Local\Temp\ipykernel_11000\777626093.py:62: RuntimeWarning: All-NaN slice encountered
  values['relative_intensity4']=np.column_stack([np.nanmax(joint,axis=1),np.sort(joint,axis=1)[:,-2],np.nanmean(joint,axis=1),(joint>1).sum(axis=1)])
C:\Users\hayeon\AppData\Local\Temp\ipykernel_11000\777626093.py:62: RuntimeWarning: Mean of empty slice
  values['relative_intensity4']=np.column_stack([np.nanmax(joint,axis=1),np.sort(joint,axis=1)[:,-2],np.nanmean(joint,axis=1),(joint>1).sum(axis=1)])


C:\Users\hayeon\AppData\Local\Temp\ipykernel_11000\777626093.py:62: RuntimeWarning: All-NaN slice encountered
  values['relative_intensity4']=np.column_stack([np.nanmax(joint,axis=1),np.sort(joint,axis=1)[:,-2],np.nanmean(joint,axis=1),(joint>1).sum(axis=1)])
C:\Users\hayeon\AppData\Local\Temp\ipykernel_11000\777626093.py:62: RuntimeWarning: Mean of empty slice
  values['relative_intensity4']=np.column_stack([np.nanmax(joint,axis=1),np.sort(joint,axis=1)[:,-2],np.nanmean(joint,axis=1),(joint>1).sum(axis=1)])


C:\Users\hayeon\AppData\Local\Temp\ipykernel_11000\777626093.py:62: RuntimeWarning: All-NaN slice encountered
  values['relative_intensity4']=np.column_stack([np.nanmax(joint,axis=1),np.sort(joint,axis=1)[:,-2],np.nanmean(joint,axis=1),(joint>1).sum(axis=1)])
C:\Users\hayeon\AppData\Local\Temp\ipykernel_11000\777626093.py:62: RuntimeWarning: Mean of empty slice
  values['relative_intensity4']=np.column_stack([np.nanmax(joint,axis=1),np.sort(joint,axis=1)[:,-2],np.nanmean(joint,axis=1),(joint>1).sum(axis=1)])


C:\Users\hayeon\AppData\Local\Temp\ipykernel_11000\777626093.py:62: RuntimeWarning: All-NaN slice encountered
  values['relative_intensity4']=np.column_stack([np.nanmax(joint,axis=1),np.sort(joint,axis=1)[:,-2],np.nanmean(joint,axis=1),(joint>1).sum(axis=1)])
C:\Users\hayeon\AppData\Local\Temp\ipykernel_11000\777626093.py:62: RuntimeWarning: Mean of empty slice
  values['relative_intensity4']=np.column_stack([np.nanmax(joint,axis=1),np.sort(joint,axis=1)[:,-2],np.nanmean(joint,axis=1),(joint>1).sum(axis=1)])


C:\Users\hayeon\AppData\Local\Temp\ipykernel_11000\777626093.py:62: RuntimeWarning: All-NaN slice encountered
  values['relative_intensity4']=np.column_stack([np.nanmax(joint,axis=1),np.sort(joint,axis=1)[:,-2],np.nanmean(joint,axis=1),(joint>1).sum(axis=1)])
C:\Users\hayeon\AppData\Local\Temp\ipykernel_11000\777626093.py:62: RuntimeWarning: Mean of empty slice
  values['relative_intensity4']=np.column_stack([np.nanmax(joint,axis=1),np.sort(joint,axis=1)[:,-2],np.nanmean(joint,axis=1),(joint>1).sum(axis=1)])


...

C:\Users\hayeon\AppData\Local\Temp\ipykernel_11000\777626093.py:62: RuntimeWarning: All-NaN slice encountered
  values['relative_intensity4']=np.column_stack([np.nanmax(joint,axis=1),np.sort(joint,axis=1)[:,-2],np.nanmean(joint,axis=1),(joint>1).sum(axis=1)])
C:\Users\hayeon\AppData\Local\Temp\ipykernel_11000\777626093.py:62: RuntimeWarning: Mean of empty slice
  values['relative_intensity4']=np.column_stack([np.nanmax(joint,axis=1),np.sort(joint,axis=1)[:,-2],np.nanmean(joint,axis=1),(joint>1).sum(axis=1)])


C:\Users\hayeon\AppData\Local\Temp\ipykernel_11000\777626093.py:62: RuntimeWarning: All-NaN slice encountered
  values['relative_intensity4']=np.column_stack([np.nanmax(joint,axis=1),np.sort(joint,axis=1)[:,-2],np.nanmean(joint,axis=1),(joint>1).sum(axis=1)])
C:\Users\hayeon\AppData\Local\Temp\ipykernel_11000\777626093.py:62: RuntimeWarning: Mean of empty slice
  values['relative_intensity4']=np.column_stack([np.nanmax(joint,axis=1),np.sort(joint,axis=1)[:,-2],np.nanmean(joint,axis=1),(joint>1).sum(axis=1)])


C:\Users\hayeon\AppData\Local\Temp\ipykernel_11000\777626093.py:62: RuntimeWarning: All-NaN slice encountered
  values['relative_intensity4']=np.column_stack([np.nanmax(joint,axis=1),np.sort(joint,axis=1)[:,-2],np.nanmean(joint,axis=1),(joint>1).sum(axis=1)])
C:\Users\hayeon\AppData\Local\Temp\ipykernel_11000\777626093.py:62: RuntimeWarning: Mean of empty slice
  values['relative_intensity4']=np.column_stack([np.nanmax(joint,axis=1),np.sort(joint,axis=1)[:,-2],np.nanmean(joint,axis=1),(joint>1).sum(axis=1)])


completed rg3_R5 OR


         model  TP  FP  FN  TN       F1
      fixed_OR   5  10   3 556 0.434783
   fixed_joint   2   4   6 562 0.285714
 fixed_thermal   3   7   5 559 0.333333
inner_selected   3   8   5 558 0.315789
 block selected  inner_thermal_F1  inner_joint_F1  inner_OR_F1  thermal_threshold  joint_threshold
rg3_R1       OR          0.250000        0.250000     0.363636           0.657474         0.930317
rg3_R2       OR          0.153846        0.285714     0.315789           0.636561         0.930737
rg3_R3  thermal          0.000000        0.000000     0.000000           0.614674         0.980754
rg3_R4       OR          0.153846        0.285714     0.315789           0.638344         0.906609
rg3_R5       OR          0.181818        0.333333     0.352941           0.632388         0.987587
